## Fase #3: Preparación de los Datos / Data Preparation
La fase de preparación de datos abarca todas las actividades para construir el conjunto de datos final (datos que se introducirán en las herramientas de modelado) a partir de los datos brutos iniciales. Es probable que las tareas de preparación de datos se realicen varias veces y no en ningún orden prescrito. Las tareas incluyen la selección de tablas, registros y atributos, así como la transformación y limpieza de datos para herramientas de modelado.
* Enumerá y describí las variables relevantes que son de tu interés para el estudio.Por ejemplo, edad: es la edad en años cumplidos al 31 de diciembre de 2024, es una variable continua discreta.
* Armá una tabla que resuma el análisis estadístico descriptivo y los gráficos que realizarás para entender las características de los datos.

In [ ]:
# @title Importaciones iniciales

import pandas as pd

from comun import RUTA_DATOS, RUTA_DATOS_LIMPIOS


## 3.1. Tratamiento de valores faltantes e inválidos

In [2]:
# @title Carga de dataset

df_datos_energia = pd.read_csv(RUTA_DATOS)

### 3.1.1 Tratamiento en `potencia_equipos_fractura_hp`

La variable `potencia_equipos_fractura_hp` representa la potencia de los equipos de fractura hidráulica, expresada en HP.

Durante el análisis exploratorio se identificaron 51 valores nulos explícitos y 154 registros con valor cero. Dado que una operación de fractura hidráulica requiere equipos que suministren potencia, estos ceros se consideran potencialmente inválidos y deben tratarse como datos faltantes, previa validación de su significado.

Para esta primera versión del dataset, se eliminarán los registros en los que la potencia sea nula o igual a cero. Se evita imputar estos valores debido a que los faltantes presentan una concentración por operadora y período histórico, por lo que una imputación global podría introducir sesgos.

Esta decisión reduce la cantidad de observaciones disponibles y podría afectar la representatividad del conjunto de datos. Por ello, deberá revisarse durante la evaluación del modelo.

**Criterio de limpieza:** conservar únicamente registros con valores de potencia mayores que cero.

In [3]:
print("Potencia faltante:", df_datos_energia["potencia_equipos_fractura_hp"].isna().sum())


np.int64(4983)

In [4]:
# Cantidad de registros antes de la limpieza
filas_antes = len(df_datos_energia)


# Eliminar registros con potencia nula o igual a cero
df_limpio = df_datos_energia[df_datos_energia["potencia_equipos_fractura_hp"].notna() & (df_datos_energia["potencia_equipos_fractura_hp"] > 0)].copy()

# Resumen de la limpieza
filas_despues = len(df_limpio)

print(f"Filas antes: {filas_antes}")
print(f"Filas después: {filas_despues}")
print(f"Filas eliminadas: {filas_antes - filas_despues}")


Filas antes: 4983
Filas después: 4778
Filas eliminadas: 205


### 3.1.2. Tratamiento en `tipo_reservorio`

Durante el análisis exploratorio se identificaron 35 registros sin información en `tipo_reservorio`.

Como no se dispone de evidencia suficiente para determinar la clasificación correcta de esos registros, los valores faltantes se reemplazarán por la categoría `DESCONOCIDO`.

Esta estrategia permite conservar las observaciones sin imputar una clasificación arbitraria. La categoría se tratará posteriormente como una categoría válida de una variable cualitativa durante la preparación para el modelado.

**Criterio de limpieza:** reemplazar los valores nulos por `DESCONOCIDO`.

In [5]:
df_limpio["tipo_reservorio"] = df_limpio["tipo_reservorio"].fillna("DESCONOCIDO")


### 3.1.3. Tratamiento en `subtipo_reservorio`

Durante el análisis exploratorio se identificaron 973 registros sin información en `subtipo_reservorio`. La investigación realizada en la fase de comprensión de los datos determinó que la mayoría de estos faltantes son estructurales: corresponden a reservorios convencionales, para los cuales la clasificación shale/tight no aplica.

Por este motivo, los valores faltantes se tratarán según el tipo de reservorio:

- Para reservorios convencionales, se asignará `NO APLICA`.
- Para reservorios no convencionales cuyo subtipo se desconoce, se asignará `DESCONOCIDO`.
- Si falta también el tipo de reservorio, se conservará `DESCONOCIDO` en el subtipo, ya que no existe información suficiente para determinar si la clasificación aplica.

Esta distinción evita imputar categorías sin fundamento y permite conservar los registros.

**Criterio de limpieza:** utilizar `tipo_reservorio` para diferenciar los faltantes estructurales de los faltantes informativos.

**Comprobación exploratoria:** las siguientes consultas sobre los datos originales muestran los registros sin subtipo en Vaca Muerta y el reparto observado por formación. El conteo de categorías omite los valores faltantes de forma predeterminada; por eso no permite asignar SHALE a los casos sin dato. Se conserva el criterio de imputación según `tipo_reservorio` explicado arriba.


In [6]:
df_datos_energia[df_datos_energia["subtipo_reservorio"].isna() & df_datos_energia["formacion_productiva"].eq("vaca muerta")]

,id_base_fractura_adjiv,idpozo,sigla,cuenca,areapermisoconcesion,yacimiento,formacion_productiva,tipo_reservorio,subtipo_reservorio,longitud_rama_horizontal_m,...,fecha_data,anio_if,mes_if,anio_ff,mes_ff,anio_carga,mes_carga,empresa_informante,mes,anio
3073,3708,163371,TPT.Nq.FP-1442(h),NEUQUINA,FORTIN DE PIEDRA,FORTIN DE PIEDRA,vaca muerta,NO CONVENCIONAL,NaN,0.0,...,2022-08-11 11:34:59.633614,2022,5,2022,6,2022,8,TECPETROL S.A.,5,2022
3103,3738,162390,TPT.Nq.FP-1326(h),NEUQUINA,FORTIN DE PIEDRA,FORTIN DE PIEDRA,vaca muerta,NO CONVENCIONAL,NaN,0.0,...,2022-08-11 11:34:59.633614,2021,5,2021,6,2022,8,TECPETROL S.A.,5,2021
3636,4334,164943,VIS.Nq.BPO-2803(h),NEUQUINA,BAJADA DEL PALO OESTE,BAJADA DEL PALO OESTE,vaca muerta,NO DISCRIMINADO,NaN,3220.0,...,2024-03-13 09:17:07.422643,2024,1,2024,2,2024,3,VISTA ENERGY ARGENTINA SAU,1,2024


In [7]:
df_datos_energia[
	(
		# df_datos_energia["subtipo_reservorio"].isna() &
		df_datos_energia["formacion_productiva"].eq("vaca muerta")
	)
]["subtipo_reservorio"].value_counts()

subtipo_reservorio
SHALE    3089
TIGHT       2
Name: count, dtype: int64

In [8]:
df_datos_energia.groupby("formacion_productiva")["subtipo_reservorio"].value_counts()

formacion_productiva    subtipo_reservorio
ANITA                   TIGHT                    4
agrio                   TIGHT                   51
                        SHALE                    1
chachao                 SHALE                    4
formación improductiva  TIGHT                    2
lajas                   TIGHT                  296
los molles              TIGHT                   85
                        SHALE                    7
magallanes              TIGHT                  195
mina el carmen          TIGHT                    1
mulichinco              TIGHT                  150
palermo aike            SHALE                    1
pozo d-129              SHALE                    4
                        TIGHT                    2
precuyo                 TIGHT                   41
punta rosada            TIGHT                   74
quintuco                SHALE                    1
vaca muerta             SHALE                 3089
                        TIGHT          

In [ ]:
# @title Completar categorías de subtipo según el tipo de reservorio
convencional_sin_subtipo = df_limpio["tipo_reservorio"].eq("CONVENCIONAL") & df_limpio["subtipo_reservorio"].isna()
df_limpio.loc[convencional_sin_subtipo, "subtipo_reservorio"] = "NO APLICA"
df_limpio["subtipo_reservorio"] = df_limpio["subtipo_reservorio"].fillna("DESCONOCIDO")
print(f"Subtipos convencionales marcados NO APLICA: {convencional_sin_subtipo.sum():,}")
print(f"Subtipos sin dato restantes: {df_limpio.subtipo_reservorio.isna().sum():,}")


## 3.2. Tratamiento de valores atípicos (outliers)

La fase 2 detecta dispersión y posibles valores extremos en `agua_inyectada_m3`. No se eliminan registros solo por un umbral estadístico: un volumen alto podría corresponder a una operación real de mayor escala. Se excluyen únicamente los valores sin sentido físico documentados (por ejemplo, potencia nula o cero). Los casos extremos y su efecto sobre el error se revisarán en la fase 5.


## 3.3 Eliminación de variables temporales redundantes

Se verificó la consistencia de las ocho variables temporales derivadas con sus respectivas fechas de origen:

- `anio_if` y `mes_if` con `fecha_inicio_fractura`.
- `anio_ff` y `mes_ff` con `fecha_fin_fractura`.
- `anio_carga` y `mes_carga` con `fecha_data`.
- `anio` y `mes` con `fecha_inicio_fractura`.

La verificación no detectó discrepancias entre los valores derivados y sus fechas de origen.

Dado que estas columnas contienen información redundante, se decide eliminarlas del conjunto de datos para reducir la cantidad de variables y simplificar su manipulación durante las siguientes etapas del proyecto.

Se conservan las fechas originales, que podrán convertirse a tipo `datetime` y utilizarse para derivar componentes temporales mediante pandas cuando sea necesario.

Esta transformación no implica descartar la información cronológica, sino evitar mantener representaciones duplicadas de ella.

**Resultado:** se eliminan `anio_if`, `mes_if`, `anio_ff`, `mes_ff`, `anio_carga`, `mes_carga`, `anio` y `mes`.

In [9]:
# @title Eliminacion de columnas derivadas
columnas_a_eliminar = [
	"anio_if",
	"mes_if",
	"anio_ff",
	"mes_ff",
	"anio_carga",
	"mes_carga",
	"mes",
	"anio",
]

# Eliminar únicamente las columnas existentes.
df_limpio = df_limpio.drop(columns=[columna for columna in columnas_a_eliminar if columna in df_limpio.columns])

# Conservar las fechas originales como datetime.
for columna in [
	"fecha_inicio_fractura",
	"fecha_fin_fractura",
	"fecha_data",
]:
	if columna in df_limpio.columns:
		df_limpio[columna] = pd.to_datetime(
			df_limpio[columna],
			errors="coerce",
		)

print("Columnas eliminadas:", [c for c in columnas_a_eliminar if c not in df_limpio.columns])
print("Columnas restantes:", df_limpio.columns)


Columnas eliminadas: ['anio_if', 'mes_if', 'anio_ff', 'mes_ff', 'anio_carga', 'mes_carga', 'mes', 'anio']
Columnas restantes: Index(['id_base_fractura_adjiv', 'idpozo', 'sigla', 'cuenca',
       'areapermisoconcesion', 'yacimiento', 'formacion_productiva',
       'tipo_reservorio', 'subtipo_reservorio', 'longitud_rama_horizontal_m',
       'cantidad_fracturas', 'tipo_terminacion', 'arena_bombeada_nacional_tn',
       'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3',
       'presion_maxima_psi', 'potencia_equipos_fractura_hp',
       'fecha_inicio_fractura', 'fecha_fin_fractura', 'fecha_data',
       'empresa_informante'],
      dtype='str')


### 3.4. Selección de la población de estudio

La fase 1 define el análisis de reservorios **no convencionales de la cuenca Neuquina**. La fase 2 muestra diferencias fuertes de volumen entre cuencas, tipos de pozo y subtipos; mezclar esos grupos dificultaría interpretar el modelo. Se seleccionan operaciones con `cuenca = NEUQUINA`, `tipo_reservorio = NO CONVENCIONAL`, `subtipo_reservorio = SHALE` y `longitud_rama_horizontal_m > 0`. La longitud positiva es el criterio operativo de pozo horizontal: la fase 2 documenta que cero corresponde a un pozo vertical. Se conserva `formacion_productiva` sin imponer que sea Vaca Muerta. La cantidad de filas excluidas se calcula abajo, sin asumirla.

Esta decisión restringe las conclusiones al grupo seleccionado. El agua registrada en cero se identificó en la fase 2 como inválida para entrenar y se excluye aquí. Fechas posteriores a la revisión del 28/10/2026 se dejan fuera por no ser operaciones históricas observables a esa fecha.


In [ ]:
# @title Delimitar el dominio y excluir objetivos inválidos
import numpy as np

fecha_revision = pd.Timestamp("2026-10-28")
filas_preparadas = len(df_limpio)
alcance = (
	df_limpio["cuenca"].eq("NEUQUINA")
	& df_limpio["tipo_reservorio"].eq("NO CONVENCIONAL")
	& df_limpio["subtipo_reservorio"].eq("SHALE")
	& df_limpio["longitud_rama_horizontal_m"].gt(0)
)
agua = pd.to_numeric(df_limpio["agua_inyectada_m3"], errors="coerce")
fecha_inicio = pd.to_datetime(df_limpio["fecha_inicio_fractura"], errors="coerce")
valido = alcance & agua.gt(0) & np.isfinite(agua) & fecha_inicio.notna() & fecha_inicio.le(fecha_revision)
print(f"Registros tras limpieza inicial: {filas_preparadas:,}")
print(f"Dentro del alcance geológico y operativo: {alcance.sum():,}")
print(f"Agua nula/no positiva dentro del alcance: {(alcance & ~agua.gt(0)).sum():,}")
print(f"Fechas posteriores a la revisión dentro del alcance: {(alcance & fecha_inicio.gt(fecha_revision)).sum():,}")
df_limpio = df_limpio.loc[valido].copy()
print(f"Registros finales para modelado: {len(df_limpio):,}")
print("Formaciones en el conjunto final:")
print(df_limpio["formacion_productiva"].value_counts())


### 3.5. Variables seleccionadas para modelado

| Variable | Rol en la fase 4 | Tipo | Definición operativa |
|---|---|---|---|
| `agua_inyectada_m3` | Objetivo | Cuantitativa continua | Volumen de agua utilizado en la fractura, en m³. |
| `longitud_rama_horizontal_m` | Entrada | Cuantitativa continua | Longitud horizontal del pozo, en metros. |
| `cantidad_fracturas` | Entrada condicional | Cuantitativa discreta | Cantidad de fracturas o etapas informadas; solo aplicable para una predicción previa si estaba planificada. |
| `tipo_terminacion` | Entrada condicional | Cualitativa nominal | Tipo de terminación informado; solo aplicable si estaba definido previamente. |
| `tipo_reservorio` | Delimitación | Cualitativa nominal | Fijado en NO CONVENCIONAL para este estudio. |
| `subtipo_reservorio` | Delimitación | Cualitativa nominal | Fijado en SHALE para este estudio. |
| `cuenca` | Delimitación | Cualitativa nominal | Fijada en NEUQUINA para este estudio. |
| `formacion_productiva` | Descriptiva | Cualitativa nominal | Formación geológica productiva; no se usa como entrada por el predominio de Vaca Muerta. |
| `potencia_equipos_fractura_hp` | Candidata no usada | Cuantitativa continua | Potencia de los equipos, en HP; el CSV no documenta si estaba disponible al planificar. |

**Disponibilidad temporal de las entradas.** La [Secretaría de Energía describe el tablero de fracturas](https://estadisticasenergia.mecon.gob.ar/superset/dashboard/52/?native_filters=%28%29&standalone=true) como elaborado a partir de declaraciones juradas de las empresas. En el conjunto preparado, `fecha_data` es posterior a `fecha_inicio_fractura` en 2.631 de 2.632 registros (en uno es anterior); es la fecha de carga del registro, no una fecha de aprobación del plan. No hay columnas con la cantidad de etapas **planificada** ni con la fecha en que se definió el tipo de terminación. Por lo tanto, estos datos no permiten comprobar que `cantidad_fracturas` y `tipo_terminacion` fueran conocidos antes de cada operación. La fase 4 puede estudiar su relación con el agua histórica; una predicción previa real requiere disponer de los valores planificados y validar que sean equivalentes a las variables publicadas.

**Consideraciones para el modelado**

- No utilizar inicialmente `empresa_informante` como predictor; conservarla para posibles análisis de sesgos y diferencias de reporte.
- Excluir variables que describen resultados de la operación, como arena bombeada, CO₂ inyectado, presión máxima efectiva y duración real de la fractura, dado que el objetivo es estimar el agua antes de iniciar la operación.
- Se excluyeron los registros con `agua_inyectada_m3 = 0` en la selección del conjunto preparado.
- Se trataron los valores faltantes de `subtipo_reservorio`, distinguiendo los casos no aplicables de los desconocidos.
- **Alcance del conjunto preparado:** se restringe a operaciones shale con rama horizontal positiva en la cuenca Neuquina, de acuerdo con el objetivo de la fase 1 y el análisis de la fase 2. La formación productiva no se usa como filtro: se conserva como variable descriptiva y posible predictor. El modelo no se generaliza a pozos verticales, tight ni otras cuencas.

In [ ]:
# @title Guardado del conjunto preparado para la Fase 4
df_limpio.to_csv(RUTA_DATOS_LIMPIOS, index=False)
print(f"Guardadas {len(df_limpio):,} operaciones en df_limpio.csv")
